In [ ]:

# Cell 1 — Install dependencies and clone repositories

!pip install --upgrade pip
!pip install torch torchinfo tqdm numpy scikit-learn matplotlib pandas toml

!rm -rf /content/KAN-AD
!rm -rf /content/datasets

!git clone https://github.com/CSTCloudOps/KAN-AD.git /content/KAN-AD
!git clone https://github.com/CSTCloudOps/datasets.git /content/datasets

!rm -rf /content/KAN-AD/datasets
!mv /content/datasets /content/KAN-AD/datasets

%cd /content/KAN-AD

Cloning into '/content/KAN-AD'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 61.77 KiB | 4.41 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Cloning into '/content/datasets'...
remote: Enumerating objects: 4503, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 4503 (delta 2), reused 0 (delta 0), pack-reused 4496 (from 3)
Receiving objects: 100% (4503/4503), 262.52 MiB | 18.93 MiB/s, done.
Resolving deltas: 100% (1962/1962), done.
Updating files: 100% (5564/5564), done.
/content/KAN-AD


In [ ]:

# Cell 2 — Install EasyTSAD, fix import syntax issue, and configure paths

!pip install git+https://github.com/CSTCloudOps/EasyTSAD.git
!sed -i 's/TSData,*/TSData/g' /usr/local/lib/python3.*/dist-packages/EasyTSAD/DataFactory/__init__.py || true
!grep -n "TSData" /usr/local/lib/python3.*/dist-packages/EasyTSAD/DataFactory/__init__.py | head -n 20

import os
import sys
import glob

REPO_ROOT = "/content/KAN-AD"
DATA_ROOT = "/content/KAN-AD/datasets"

sys.path.insert(0, REPO_ROOT)

candidates = glob.glob("/content/KAN-AD/**/kanad/kanad.py", recursive=True)
print("Found KAN-AD candidates:", candidates)
assert candidates, "Could not find kanad/kanad.py inside the cloned KAN-AD repo."

KANAD_PY = sorted(candidates, key=len)[0]
KANAD_PKG_DIR = os.path.dirname(KANAD_PY)
KANAD_ROOT_DIR = os.path.dirname(KANAD_PKG_DIR)

sys.path.insert(0, KANAD_ROOT_DIR)

print("KANAD_PKG_DIR =", KANAD_PKG_DIR)
print("KANAD_ROOT_DIR =", KANAD_ROOT_DIR)
print("sys.path[:4] =", sys.path[:4])

  Cloning https://github.com/CSTCloudOps/EasyTSAD.git to /tmp/pip-req-build-f8gbmupv
  Running command git clone --filter=blob:none --quiet https://github.com/CSTCloudOps/EasyTSAD.git /tmp/pip-req-build-f8gbmupv
  Resolved https://github.com/CSTCloudOps/EasyTSAD.git to commit f4f3e594e7c9218a4d6fad29f40f72b5f1561fe7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
1:from .TSData import TSData
2:from .MTSData import MTSData
Found KAN-AD candidates: ['/content/KAN-AD/kanad/kanad.py']
KANAD_PKG_DIR = /content/KAN-AD/kanad
KANAD_ROOT_DIR = /content/KAN-AD
sys.path[:4] = ['/content/KAN-AD', '/content/KAN-AD', '/content', '/env/python']


In [ ]:

# Cell 3 — Imports and sanity check

import json
import math
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import tqdm

from torch.utils.data import Dataset, DataLoader

from EasyTSAD.Controller import TSADController
from EasyTSAD.DataFactory import TSData
from EasyTSAD.Methods import BaseMethod
from EasyTSAD.Evaluations.Protocols import (
    EventF1PA,
    PointF1PA,
    PointKthF1PA,
    PointAuprcPA,
)

from kanad import KANAD
from kanad.kanad import KANADModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("KANAD imported:", KANAD)
print("KANADModel imported:", KANADModel)
print("TSADController imported:", TSADController)

Device: cuda
KANAD imported: <class 'kanad.kanad.KANAD'>
KANADModel imported: <class 'kanad.kanad.KANADModel'>
TSADController imported: <class 'EasyTSAD.Controller.TSADController.TSADController'>


In [ ]:
# Cell 4 — Define the Wavelet-based KAN-AD backbone helpers + sensor attention

class PeriodicIndexBasisLayer(nn.Module):
    def __init__(self, order: int, window: int):
        super().__init__()
        self.order = int(order)
        self.window = int(window)
        idx = torch.arange(self.window, dtype=torch.float32)
        self.register_buffer("idx", idx)

    def forward(self, x):
        B, W = x.shape
        feats = []
        t = self.idx.to(x.device)
        for n in range(1, self.order + 1):
            feats.append(torch.sin(2.0 * math.pi * n * t / self.window).view(1, 1, W).repeat(B, 1, 1))
            feats.append(torch.cos(2.0 * math.pi * n * t / self.window).view(1, 1, W).repeat(B, 1, 1))
        return torch.cat(feats, dim=1) if feats else torch.empty(B, 0, W, device=x.device)


class WaveletBasisLayer(nn.Module):
    """
    Morlet wavelet family (commonly used in KAN-AD wavelet variants)
    ψ(x) = cos(ωx) * exp(-x^2 / s^2)
    """

    def __init__(self, n_scales: int = 4):
        super().__init__()
        self.n_scales = int(n_scales)

        scales = torch.linspace(0.5, 2.0, self.n_scales)
        self.register_buffer("scales", scales)

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, W)
        s = self.scales.view(1, -1, 1)  # (1, S, 1)

        # Morlet wavelet
        wavelet = torch.cos(x / s) * torch.exp(-(x ** 2) / (s ** 2 + 1e-8))
        return wavelet


class WaveletKANADModel(nn.Module):
    """
    Wavelet-based KAN-AD backbone
    """

    def __init__(self, window: int, order: int = 2, family_channels: int = 4):
        super().__init__()

        self.window = int(window)
        self.order = int(order)
        self.family_channels = int(family_channels)

        # 🔥 Wavelet replaces Fourier/RBF
        self.family_layer = WaveletBasisLayer(n_scales=self.family_channels)
        self.family_out_channels = self.family_channels

        self.periodic_layer = PeriodicIndexBasisLayer(order=self.order, window=self.window)
        self.periodic_out_channels = 2 * self.order

        self.channels = self.family_out_channels + self.periodic_out_channels + 1

        self.init_conv = nn.Conv1d(self.channels, self.channels, kernel_size=3, padding=1, bias=False)
        self.inner_conv = nn.Conv1d(self.channels, self.channels, kernel_size=3, padding=1, bias=False)
        self.out_conv = nn.Conv1d(self.channels, 1, kernel_size=1, bias=False)
        self.final_conv = nn.Conv1d(1, 1, kernel_size=self.window)

        self.bn1 = nn.BatchNorm1d(self.channels)
        self.bn2 = nn.BatchNorm1d(self.channels)
        self.bn3 = nn.BatchNorm1d(1)
        self.act = nn.GELU()

    def forward_feature(self, x: torch.Tensor):
        raw = x.unsqueeze(1)

        # 🔥 Wavelet basis
        family = self.family_layer(x)

        periodic = self.periodic_layer(x)

        ff = torch.cat([family, periodic, raw], dim=1)

        res0 = raw
        res1 = ff

        ff = self.act(self.bn1(self.init_conv(ff)))
        ff = self.act(self.bn2(self.inner_conv(ff) + res1))
        ff = self.act(self.bn3(self.out_conv(ff) + res0))

        return ff

    def forward_head(self, ff: torch.Tensor):
        return self.final_conv(ff).squeeze(1)

    def forward(self, x: torch.Tensor):
        ff = self.forward_feature(x)
        return self.forward_head(ff)


# ✅ KEEP ATTENTION EXACTLY SAME
class SensorSelfAttention(nn.Module):
    def __init__(self, n_features: int, d_model: int = 16, n_heads: int = 2, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_features = int(n_features)
        self.embed = nn.Linear(1, d_model)
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.out = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        B, T, F = x.shape

        xt = x.reshape(B * T, F, 1)
        z = self.embed(xt)

        if return_attn:
            z_attn, attn = self.mha(z, z, z, need_weights=True, average_attn_weights=True)
            attn_avg = attn.mean(dim=0)
        else:
            z_attn, _ = self.mha(z, z, z, need_weights=False)
            attn_avg = None

        y = self.out(z_attn).squeeze(-1)
        x_mix = y.reshape(B, T, F)

        return x_mix, attn_avg


print("Wavelet FastKANADModel + SensorSelfAttention ready.")

Wavelet FastKANADModel + SensorSelfAttention ready.


In [ ]:
# Cell 5 — Build MSL holdout dataset + save calibration arrays outside EasyTSAD tree

import os
import shutil
import numpy as np

# -----------------------------
# Basic paths / names
# -----------------------------
ROOT_DIR = "/content/KAN-AD"
DATA_ROOT = os.path.join(ROOT_DIR, "datasets", "MTS")

ORIG_DATASET = "MSL"                       # ✅ CHANGED
CUSTOM_DATASET = "MSL_HybridHoldout"       # ✅ CHANGED

orig_curve_dir = os.path.join(DATA_ROOT, ORIG_DATASET, "AllInOne")
custom_dataset_dir = os.path.join(DATA_ROOT, CUSTOM_DATASET)
custom_curve_dir = os.path.join(custom_dataset_dir, "AllInOne")

print("Original MSL curve dir:", orig_curve_dir)

# -----------------------------
# Load original MSL arrays
# -----------------------------
train = np.load(os.path.join(orig_curve_dir, "train.npy"))
test = np.load(os.path.join(orig_curve_dir, "test.npy"))

train_label_path = os.path.join(orig_curve_dir, "train_label.npy")
test_label_path = os.path.join(orig_curve_dir, "test_label.npy")

if os.path.exists(train_label_path):
    train_label = np.load(train_label_path)
else:
    train_label = np.zeros(len(train), dtype=np.int64)

if os.path.exists(test_label_path):
    test_label = np.load(test_label_path)
else:
    raise FileNotFoundError(f"Missing test labels: {test_label_path}")

print("train shape:", train.shape)
print("test shape :", test.shape)
print("train_label shape:", train_label.shape, " positives:", int(train_label.sum()))
print("test_label shape :", test_label.shape, " positives:", int(test_label.sum()))

# -----------------------------
# Calibration split (⚠️ MSL-specific tweak)
# -----------------------------
test_len = len(test)

# 🔥 MSL needs slightly larger calibration (anomalies are sparse)
default_calib_end = int(0.30 * test_len)   # ✅ CHANGED (was 20%)

positive_idx = np.where(test_label > 0)[0]
if len(positive_idx) == 0:
    raise ValueError("MSL test labels contain no anomalies.")

# ensure we include enough anomaly segments
last_pos = int(positive_idx[min(len(positive_idx) - 1, max(5, int(0.20 * len(positive_idx))))])

calib_end = max(default_calib_end, last_pos + 1)
calib_end = min(calib_end, test_len - 1)

calib_arr = test[:calib_end].copy()
calib_label = test_label[:calib_end].copy()

final_test = test[calib_end:].copy()
final_test_label = test_label[calib_end:].copy()

print("\nChosen calibration end index:", calib_end)
print("Calibration shape:", calib_arr.shape, " positives:", int(calib_label.sum()))
print("Final test shape :", final_test.shape, " positives:", int(final_test_label.sum()))

if calib_label.sum() == 0:
    raise ValueError("Calibration split still has no anomalies.")

# -----------------------------
# Rebuild dataset tree
# -----------------------------
if os.path.exists(custom_dataset_dir):
    shutil.rmtree(custom_dataset_dir)

os.makedirs(custom_curve_dir, exist_ok=True)

np.save(os.path.join(custom_curve_dir, "train.npy"), train)
np.save(os.path.join(custom_curve_dir, "train_label.npy"), train_label)
np.save(os.path.join(custom_curve_dir, "test.npy"), final_test)
np.save(os.path.join(custom_curve_dir, "test_label.npy"), final_test_label)

# copy metadata
for extra_name in ["info.json", "meta.json"]:
    src = os.path.join(orig_curve_dir, extra_name)
    dst = os.path.join(custom_curve_dir, extra_name)
    if os.path.exists(src):
        shutil.copy2(src, dst)

# -----------------------------
# Save calibration arrays
# -----------------------------
RUNTIME_DIR = os.path.join(ROOT_DIR, "kanad_runtime")
os.makedirs(RUNTIME_DIR, exist_ok=True)

CALIB_DIR = os.path.join(RUNTIME_DIR, f"{CUSTOM_DATASET}_calibration")
if os.path.exists(CALIB_DIR):
    shutil.rmtree(CALIB_DIR)
os.makedirs(CALIB_DIR, exist_ok=True)

np.save(os.path.join(CALIB_DIR, "calib.npy"), calib_arr)
np.save(os.path.join(CALIB_DIR, "calib_label.npy"), calib_label)

# -----------------------------
# Sanity checks
# -----------------------------
print("\nCreated EasyTSAD dataset at:", custom_curve_dir)
print("Saved files:", sorted(os.listdir(custom_curve_dir)))

print("\nCalibration dir:", CALIB_DIR)
print("Calibration files:", sorted(os.listdir(CALIB_DIR)))

assert os.path.exists(os.path.join(custom_curve_dir, "train.npy"))
assert os.path.exists(os.path.join(custom_curve_dir, "test.npy"))
assert os.path.exists(os.path.join(custom_curve_dir, "test_label.npy"))
assert os.path.exists(os.path.join(CALIB_DIR, "calib.npy"))
assert os.path.exists(os.path.join(CALIB_DIR, "calib_label.npy"))

print("\nCell 5 completed successfully.")

Original MSL curve dir: /content/KAN-AD/datasets/MTS/MSL/AllInOne
train shape: (58317, 55)
test shape : (73729, 55)
train_label shape: (58317,)  positives: 0
test_label shape : (73729,)  positives: 7766

Chosen calibration end index: 22118
Calibration shape: (22118, 55)  positives: 2608
Final test shape : (51611, 55)  positives: 5158

Created EasyTSAD dataset at: /content/KAN-AD/datasets/MTS/MSL_HybridHoldout/AllInOne
Saved files: ['info.json', 'test.npy', 'test_label.npy', 'train.npy', 'train_label.npy']

Calibration dir: /content/KAN-AD/kanad_runtime/MSL_HybridHoldout_calibration
Calibration files: ['calib.npy', 'calib_label.npy']

Cell 5 completed successfully.


In [ ]:

# Cell 6 — Utility functions: segmentation, calibration F1 search, normalization, plotting

def contiguous_segments(binary_labels: np.ndarray):
    y = np.asarray(binary_labels).astype(int)
    segs = []
    start = None
    for i, v in enumerate(y):
        if v == 1 and start is None:
            start = i
        elif v == 0 and start is not None:
            segs.append((start, i - 1))
            start = None
    if start is not None:
        segs.append((start, len(y) - 1))
    return segs


def segment_overlap(a, b):
    return not (a[1] < b[0] or b[1] < a[0])


def event_f1_from_binary(pred_binary: np.ndarray, true_binary: np.ndarray):
    gt_segs = contiguous_segments(true_binary)
    pr_segs = contiguous_segments(pred_binary)

    if len(gt_segs) == 0 and len(pr_segs) == 0:
        return 1.0, 1.0, 1.0
    if len(gt_segs) == 0:
        return 0.0, 0.0, 0.0
    if len(pr_segs) == 0:
        return 0.0, 0.0, 0.0

    used_pr = set()
    tp = 0
    for gt in gt_segs:
        for j, pr in enumerate(pr_segs):
            if j in used_pr:
                continue
            if segment_overlap(gt, pr):
                tp += 1
                used_pr.add(j)
                break

    precision = tp / max(len(pr_segs), 1)
    recall = tp / max(len(gt_segs), 1)
    f1 = 0.0 if (precision + recall == 0) else 2 * precision * recall / (precision + recall)
    return f1, precision, recall


def best_event_f1_threshold(scores: np.ndarray, labels: np.ndarray, n_grid: int = 200):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels).astype(int)

    if labels.sum() == 0:
        return 0.0, float(np.max(scores)), 0.0, 0.0

    lo = float(np.min(scores))
    hi = float(np.max(scores))

    if not np.isfinite(lo) or not np.isfinite(hi):
        raise ValueError("Non-finite scores encountered during threshold search.")

    if hi <= lo:
        pred = (scores > lo).astype(int)
        f1, p, r = event_f1_from_binary(pred, labels)
        return f1, lo, p, r

    thresholds = np.linspace(lo, hi, n_grid)
    best = (-1.0, lo, 0.0, 0.0)

    for thr in thresholds:
        pred = (scores > thr).astype(int)
        f1, p, r = event_f1_from_binary(pred, labels)
        if f1 > best[0]:
            best = (f1, float(thr), float(p), float(r))

    return best


def sigmoid_np(x):
    x = np.clip(x, -8.0, 8.0)
    return 1.0 / (1.0 + np.exp(-x))


def zscore_sigmoid(x, mu, std, eps=1e-8):
    z = (x - mu) / (std + eps)
    return sigmoid_np(z)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def show_overlay(scores, labels, title, max_points=4000):
    n = min(len(scores), max_points)
    x = np.arange(n)
    plt.figure(figsize=(14, 4))
    plt.plot(x, scores[:n], label="score")
    if labels is not None and n > 0:
        ymax = max(1e-6, float(np.max(scores[:n])))
        plt.fill_between(x, 0, ymax, where=labels[:n].astype(bool), alpha=0.2, step="pre", label="anomaly")
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:

# Cell 7 — Window datasets for train/valid/test and calibration

class MTSWindowDataset(Dataset):
    def __init__(self, tsData, phase, window_size):
        self.window_size = int(window_size)

        if phase == "train":
            self.data = np.asarray(tsData.train)
        elif phase == "valid":
            self.data = np.asarray(tsData.valid)
        elif phase == "test":
            self.data = np.asarray(tsData.test)
        else:
            raise ValueError("phase must be train / valid / test")

        assert self.data.ndim == 2, f"Expected 2D MTS array, got {self.data.shape}"
        self.N, self.F = self.data.shape
        self.sample_num = max(self.N - self.window_size, 0)

    def __len__(self):
        return self.sample_num

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size]
        y = self.data[idx + self.window_size]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


class ArrayWindowDataset(Dataset):
    def __init__(self, data: np.ndarray, labels: np.ndarray, window_size: int):
        self.data = np.asarray(data)
        self.labels = np.asarray(labels).astype(int)
        self.window_size = int(window_size)

        assert self.data.ndim == 2, f"Expected 2D array, got {self.data.shape}"
        assert len(self.data) == len(self.labels), "Data/label length mismatch"

        self.N, self.F = self.data.shape
        self.sample_num = max(self.N - self.window_size, 0)

    def __len__(self):
        return self.sample_num

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size]
        y = self.data[idx + self.window_size]
        label = self.labels[idx + self.window_size]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long),
        )

print("Datasets ready.")

Datasets ready.


In [ ]:
# Cell 8 — Define WaveletKAN + Attention model (NO DeepSVDD ablation)

class Attn_WaveletKANAD_Holdout(BaseMethod):
    def __init__(self, params: dict) -> None:
        super().__init__()

        self.__anomaly_score = None

        self.batch_size = int(params["batch_size"])
        self.window = int(params["window"])
        self.order = int(params["order"])
        self.epochs = int(params["epochs"])
        self.lr = float(params["lr"])
        self.patience = int(params.get("patience", 6))

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.n_features = int(params.get("n_features", 51))
        self.attn_d_model = int(params.get("attn_d_model", 16))
        self.attn_heads = int(params.get("attn_heads", 2))
        self.attn_dropout = float(params.get("attn_dropout", 0.0))

        self.model = WaveletKANADModel(
            window=self.window,
            order=self.order,
        ).to(self.device)

        self.sensor_attn = SensorSelfAttention(
            n_features=self.n_features,
            d_model=self.attn_d_model,
            n_heads=self.attn_heads,
            dropout=self.attn_dropout,
        ).to(self.device)

        self.optimizer = optim.Adam(
            list(self.model.parameters()) + list(self.sensor_attn.parameters()),
            lr=self.lr
        )
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=5, gamma=0.75)
        self.mse_loss = nn.MSELoss()

        self.last_attn_matrix = None
        self.best_state = None

    def _mix_sensors(self, x, return_attn: bool = False):
        x_mix, attn_avg = self.sensor_attn(x, return_attn=return_attn)
        if attn_avg is not None:
            self.last_attn_matrix = attn_avg.detach().cpu()
        return x_mix

    def _forward_batch(self, x, target):
        B, W, F = x.shape

        x_mix = self._mix_sensors(x, return_attn=False)
        x_1d = x_mix.permute(0, 2, 1).reshape(B * F, W)
        t_1d = target.reshape(B * F, 1)

        ff = self.model.forward_feature(x_1d)
        out = self.model.forward_head(ff).reshape(B * F, 1)

        pred_loss = self.mse_loss(out, t_1d)
        pred_err = (out - t_1d).abs().reshape(B, F).max(dim=1).values

        return pred_err, pred_loss

    def _collect_scores_from_loader(self, loader):
        scores_all = []

        self.model.eval()
        self.sensor_attn.eval()

        with torch.no_grad():
            for x, y in loader:
                x = x.to(self.device)
                y = y.to(self.device)

                pred_err, _ = self._forward_batch(x, y)
                scores_all.append(pred_err.detach().cpu().numpy())

        return np.concatenate(scores_all) if scores_all else np.array([], dtype=float)

    def train_valid_phase(self, tsTrain: TSData):
        train_loader = DataLoader(
            MTSWindowDataset(tsTrain, "train", self.window),
            batch_size=self.batch_size,
            shuffle=True
        )
        valid_loader = DataLoader(
            MTSWindowDataset(tsTrain, "valid", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )

        best_valid = float("inf")
        patience_counter = 0

        for epoch in range(1, self.epochs + 1):
            self.model.train()
            self.sensor_attn.train()

            train_losses = []

            for x, y in tqdm.tqdm(train_loader, desc=f"Train {epoch}"):
                x = x.to(self.device)
                y = y.to(self.device)

                self.optimizer.zero_grad(set_to_none=True)

                _, pred_loss = self._forward_batch(x, y)

                if not torch.isfinite(pred_loss):
                    print("Skipping batch due to non-finite prediction loss")
                    continue

                pred_loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(self.model.parameters()) + list(self.sensor_attn.parameters()),
                    max_norm=5.0
                )
                self.optimizer.step()

                train_losses.append(float(pred_loss.item()))

            self.model.eval()
            self.sensor_attn.eval()

            valid_losses = []
            with torch.no_grad():
                for x, y in tqdm.tqdm(valid_loader, desc=f"Valid {epoch}"):
                    x = x.to(self.device)
                    y = y.to(self.device)

                    _, pred_loss = self._forward_batch(x, y)
                    if torch.isfinite(pred_loss):
                        valid_losses.append(float(pred_loss.item()))

            train_loss = float(np.mean(train_losses)) if train_losses else np.nan
            valid_loss = float(np.mean(valid_losses)) if valid_losses else np.nan

            print(f"Epoch {epoch} | train_loss={train_loss:.6f} | valid_loss={valid_loss:.6f}")

            self.scheduler.step()

            if np.isfinite(valid_loss) and valid_loss < best_valid:
                best_valid = valid_loss
                patience_counter = 0
                self.best_state = {
                    "model": {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()},
                    "attn": {k: v.detach().cpu().clone() for k, v in self.sensor_attn.state_dict().items()},
                }
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        assert self.best_state is not None, "No best model state was saved."
        self.model.load_state_dict(self.best_state["model"])
        self.sensor_attn.load_state_dict(self.best_state["attn"])

    def test_phase(self, tsData: TSData):
        test_loader = DataLoader(
            MTSWindowDataset(tsData, "test", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )

        scores = self._collect_scores_from_loader(test_loader)

        if len(scores) == 0:
            padded = np.zeros(len(tsData.test), dtype=float)
        else:
            prefix = np.full(self.window, scores[0], dtype=float)
            padded = np.concatenate([prefix, scores], axis=0)
            padded = padded[:len(tsData.test)]

        self.__anomaly_score = padded.astype(np.float64)

    def anomaly_score(self) -> np.ndarray:
        return self.__anomaly_score

    def param_statistic(self, save_file):
        stats = {
            "WaveletKAN_trainable_params": int(count_parameters(self.model)),
            "Attention_trainable_params": int(count_parameters(self.sensor_attn)),
            "total_trainable_params": int(
                count_parameters(self.model) + count_parameters(self.sensor_attn)
            ),
            "window": int(self.window),
            "order": int(self.order),
            "n_features": int(self.n_features),
            "attn_d_model": int(self.attn_d_model),
            "attn_heads": int(self.attn_heads),
            "attn_dropout": float(self.attn_dropout),
        }
        with open(save_file, "w") as f:
            json.dump(stats, f, indent=2)

print("Ablation model ready: WaveletKAN + Attention (without DeepSVDD).")


Ablation model ready: WaveletKAN + Attention (without DeepSVDD).


In [ ]:
# Cell 9 — Create config file for WaveletKAN + Attention ablation (NO DeepSVDD)

config_text = f"""\
[Data_Params]
preprocess = "z-score"
diff_order = 0

[Model_Params.Default]
window = 96
order = 2
batch_size = 128
epochs = 60
lr = 0.001
patience = 6

# Attention parameters
n_features = 55
attn_d_model = 16
attn_heads = 2
attn_dropout = 0.0
"""

CFG_PATH = os.path.join(KANAD_PKG_DIR, "config_attn_waveletkan_no_svdd_msl.toml")

with open(CFG_PATH, "w") as f:
    f.write(config_text)

print("Wrote config to:", CFG_PATH)
print(open(CFG_PATH).read())


Wrote config to: /content/KAN-AD/kanad/config_attn_waveletkan_no_svdd_msl.toml
[Data_Params]
preprocess = "z-score"
diff_order = 0

[Model_Params.Default]
window = 96
order = 2
batch_size = 128
epochs = 60
lr = 0.001
patience = 6

# Attention parameters
n_features = 55
attn_d_model = 16
attn_heads = 2
attn_dropout = 0.0



In [ ]:
# Cell 10 — Run the EasyTSAD experiment for ablation model

gctrl = TSADController()

gctrl.set_dataset(
    dataset_type="MTS",
    dirname="/content/KAN-AD/datasets",
    datasets=[CUSTOM_DATASET],
)

METHOD_NAME = "Attn_WaveletKANAD_Holdout"
TRAINING_SCHEMA = "naive"

print("Running ablation:", "WaveletKAN + Attention (NO DeepSVDD)")
print("Dataset:", CUSTOM_DATASET)

gctrl.run_exps(
    method=METHOD_NAME,
    training_schema=TRAINING_SCHEMA,
    cfg_path=CFG_PATH,
)

gctrl.set_evals([
    PointF1PA(),
    EventF1PA(mode="squeeze"),
    PointKthF1PA(k=5),
    PointAuprcPA(),
])

gctrl.do_evals(
    method=METHOD_NAME,
    training_schema=TRAINING_SCHEMA,
)


(2026-05-29 12:35:00,107) [INFO]: 
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚══════╝   ╚═╝          ╚═╝   ╚══════╝╚═╝  ╚═╝╚═════╝ 
                                                                      
                         
INFO:logger:
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚═════

Running ablation: WaveletKAN + Attention (NO DeepSVDD)
Dataset: MSL_HybridHoldout


(2026-05-29 12:35:00,905) [INFO]:     [Attn_WaveletKANAD_Holdout] handling dataset MSL_HybridHoldout | curve AllInOne 
INFO:logger:    [Attn_WaveletKANAD_Holdout] handling dataset MSL_HybridHoldout | curve AllInOne 
Valid 1: 100%|██████████| 455/455 [08:28<00:00,  1.12s/it]


Epoch 1 | train_loss=7692.980578 | valid_loss=7693.126800


Valid 2: 100%|██████████| 455/455 [08:13<00:00,  1.08s/it]


Epoch 2 | train_loss=7692.859456 | valid_loss=7693.069433


Valid 3: 100%|██████████| 455/455 [08:08<00:00,  1.07s/it]


Epoch 3 | train_loss=7692.727505 | valid_loss=7692.231580


Train 4:  74%|███████▍  | 338/455 [19:15<06:36,  3.39s/it]

In [ ]:

# Cell 11 — Load and display EasyTSAD evaluation results

BASE_EVAL = "/content/KAN-AD/Results/Evals"
if not os.path.exists(BASE_EVAL):
    BASE_EVAL = "/content/KAN-AD/KAN-AD/Results/Evals"

avg_files = glob.glob(os.path.join(BASE_EVAL, "**", CUSTOM_DATASET, "avg.json"), recursive=True)
all_files = glob.glob(os.path.join(BASE_EVAL, "**", CUSTOM_DATASET, "all.json"), recursive=True)

print("Found avg.json:", avg_files)
print("Found all.json:", all_files)

assert avg_files, "avg.json not found"
assert all_files, "all.json not found"

avg_path = avg_files[0]
all_path = all_files[0]

with open(avg_path, "r") as f:
    avg = json.load(f)

with open(all_path, "r") as f:
    all_scores = json.load(f)

print("\n=== AVERAGE RESULTS (holdout final test) ===")
for k, v in avg.items():
    print(f"{k}: {v}")

print("\n=== PER-SERIES RESULTS ===")
print("Number of series:", len(all_scores))
if len(all_scores) > 0:
    print("Example entry:", list(all_scores.items())[0])

Found avg.json: ['/content/KAN-AD/Results/Evals/Attn_WaveletKANAD_Holdout/naive/MSL_HybridHoldout/avg.json']
Found all.json: ['/content/KAN-AD/Results/Evals/Attn_WaveletKANAD_Holdout/naive/MSL_HybridHoldout/all.json']

=== AVERAGE RESULTS (holdout final test) ===
best f1 under pa: {'f1': 0.9242623566998679, 'precision': 0.9159992549823058, 'recall': 0.932675896074341, 'threshold': 16.50404517352581}
event-based f1 under pa with mode squeeze: {'f1': 0.23809523809523758, 'precision': 0.2631578947368421, 'recall': 0.21739130434782608, 'threshold': 171.1415249556303}
best f1 under 5-delay pa: {'f1': 0.3613822210498545, 'precision': 0.5933304460805544, 'recall': 0.25981414754409254, 'threshold': 10.804441139101982}
point-based auprc pa: 0.9357563135066677

=== PER-SERIES RESULTS ===
Number of series: 1
Example entry: ('AllInOne', {'best f1 under pa': {'f1': 0.9242623566998679, 'precision': 0.9159992549823058, 'recall': 0.932675896074341, 'threshold': 16.50404517352581}, 'event-based f1 unde

In [ ]:

# Cell 12 — Optional: inspect runtime stats and selected alpha

BASE_RUNTIME = "/content/KAN-AD/Results/RunTime"
if not os.path.exists(BASE_RUNTIME):
    BASE_RUNTIME = "/content/KAN-AD/KAN-AD/Results/RunTime"

runtime_files = glob.glob(
    os.path.join(BASE_RUNTIME, "**", METHOD_NAME, TRAINING_SCHEMA, CUSTOM_DATASET, "*.json"),
    recursive=True
)
print("Runtime files:", runtime_files[:5])

for fp in runtime_files[:5]:
    print("\n---", fp, "---")
    try:
        print(json.dumps(json.load(open(fp, "r")), indent=2)[:3000])
    except Exception as e:
        print("Could not read:", e)

Runtime files: ['/content/KAN-AD/Results/RunTime/Attn_WaveletKANAD_Holdout/naive/MSL_HybridHoldout/time.json']

--- /content/KAN-AD/Results/RunTime/Attn_WaveletKANAD_Holdout/naive/MSL_HybridHoldout/time.json ---
{
  "train_and_valid": 374.67549618699996,
  "test": 7.221069725000007
}
